In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import os

CATALOG = "healthcare_analytics"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/healthcare_analytics/bronze/synthea_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

files = [
    f for f in dbutils.fs.ls(VOLUME_PATH)
    if f.name.lower().endswith(".csv")
]

print(f"CSV files discovered: {len(files)}")

inventory = []

for file in files:
    file_name = file.name
    table_base = os.path.splitext(file_name)[0].lower()
    table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_base}_raw"

    print(f"Loading: {file_name} -> {table_name}")

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .csv(file.path)
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    inventory.append(
        (
            file_name,
            table_name,
            len(df.columns),
            file.size,
            datetime.now()
        )
    )

inventory_df = spark.createDataFrame(
    inventory,
    [
        "source_file",
        "bronze_table",
        "column_count",
        "file_size_bytes",
        "loaded_at"
    ]
)

(
    inventory_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_file_inventory"
    )
)

display(
    spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.bronze_file_inventory"
    ).orderBy(F.desc("file_size_bytes"))
)

CSV files discovered: 18
Loading: allergies.csv -> healthcare_analytics.bronze.allergies_raw
Loading: careplans.csv -> healthcare_analytics.bronze.careplans_raw
Loading: claims.csv -> healthcare_analytics.bronze.claims_raw
Loading: claims_transactions.csv -> healthcare_analytics.bronze.claims_transactions_raw
Loading: conditions.csv -> healthcare_analytics.bronze.conditions_raw
Loading: devices.csv -> healthcare_analytics.bronze.devices_raw
Loading: encounters.csv -> healthcare_analytics.bronze.encounters_raw
Loading: imaging_studies.csv -> healthcare_analytics.bronze.imaging_studies_raw
Loading: immunizations.csv -> healthcare_analytics.bronze.immunizations_raw
Loading: medications.csv -> healthcare_analytics.bronze.medications_raw
Loading: observations.csv -> healthcare_analytics.bronze.observations_raw
Loading: organizations.csv -> healthcare_analytics.bronze.organizations_raw
Loading: patients.csv -> healthcare_analytics.bronze.patients_raw
Loading: payer_transitions.csv -> healthc

source_file,bronze_table,column_count,file_size_bytes,loaded_at
claims_transactions.csv,healthcare_analytics.bronze.claims_transactions_raw,35,458140374,2026-09-07T18:33:07.505Z
observations.csv,healthcare_analytics.bronze.observations_raw,11,135870674,2026-09-07T18:33:43.251Z
imaging_studies.csv,healthcare_analytics.bronze.imaging_studies_raw,15,47788222,2026-09-07T18:33:28.081Z
claims.csv,healthcare_analytics.bronze.claims_raw,33,44019723,2026-09-07T18:33:00.552Z
procedures.csv,healthcare_analytics.bronze.procedures_raw,12,38719104,2026-09-07T18:34:05.882Z
encounters.csv,healthcare_analytics.bronze.encounters_raw,17,21377107,2026-09-07T18:33:22.817Z
medications.csv,healthcare_analytics.bronze.medications_raw,15,13218730,2026-09-07T18:33:37.647Z
payer_transitions.csv,healthcare_analytics.bronze.payer_transitions_raw,10,7069470,2026-09-07T18:33:56.424Z
conditions.csv,healthcare_analytics.bronze.conditions_raw,9,6119054,2026-09-07T18:33:12.799Z
supplies.csv,healthcare_analytics.bronze.supplies_raw,8,3968961,2026-09-07T18:34:14.342Z


In [0]:
tables = spark.sql("""
SHOW TABLES IN healthcare_analytics.bronze
""")

display(tables)

database,tableName,isTemporary
bronze,allergies_raw,false
bronze,bronze_file_inventory,false
bronze,careplans_raw,false
bronze,claims_raw,false
bronze,claims_transactions_raw,false
bronze,conditions_raw,false
bronze,devices_raw,false
bronze,encounters_raw,false
bronze,imaging_studies_raw,false
bronze,immunizations_raw,false
